# ***Assignment 9: Data Preprocessing and Feature Engineering in Machine Learning***

In [10]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

# **1. Data Exploration and Preprocessing**

In [11]:
# Load dataset[cite: 10]
df = pd.read_csv("/content/adult_with_headers (1).csv")

print("--- Data Exploration: Summary Statistics ---")
print(df.describe())

print("\n--- Data Exploration: Missing Values & Data Types ---")
print(df.info())

# In the Adult dataset, missing values are typically coded as ' ?'
df = df.replace(r"^\s*\?\s*$", np.nan, regex=True)

print("\n--- Missing Values Count per Column ---")
print(df.isnull().sum())

# Handle missing values: Impute categorical columns with mode[cite: 10]
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

--- Data Exploration: Summary Statistics ---
                age        fnlwgt  education_num  capital_gain  capital_loss  \
count  32561.000000  3.256100e+04   32561.000000  32561.000000  32561.000000   
mean      38.581647  1.897784e+05      10.080679   1077.648844     87.303830   
std       13.640433  1.055500e+05       2.572720   7385.292085    402.960219   
min       17.000000  1.228500e+04       1.000000      0.000000      0.000000   
25%       28.000000  1.178270e+05       9.000000      0.000000      0.000000   
50%       37.000000  1.783560e+05      10.000000      0.000000      0.000000   
75%       48.000000  2.370510e+05      12.000000      0.000000      0.000000   
max       90.000000  1.484705e+06      16.000000  99999.000000   4356.000000   

       hours_per_week  
count    32561.000000  
mean        40.437456  
std         12.347429  
min          1.000000  
25%         40.000000  
50%         40.000000  
75%         45.000000  
max         99.000000  

--- Data Explorat

# **2. Scaling Techniques**

In [12]:
numerical_cols = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
]

# a. Standard Scaling[cite: 10]
std_scaler = StandardScaler()
df_standard_scaled = df.copy()
df_standard_scaled[numerical_cols] = std_scaler.fit_transform(
    df[numerical_cols]
)

# b. Min-Max Scaling[cite: 10]
minmax_scaler = MinMaxScaler()
df_minmax_scaled = df.copy()
df_minmax_scaled[numerical_cols] = minmax_scaler.fit_transform(
    df[numerical_cols]
)

# **3. Encoding Techniques**

In [13]:
df_encoded = df.copy()

# Identify categories count per object column
for col in categorical_cols:
    unique_cnt = df_encoded[col].nunique()
    if unique_cnt < 5:
        # Apply One-Hot Encoding for < 5 unique categories[cite: 10]
        df_encoded = pd.get_dummies(
            df_encoded, columns=[col], drop_first=True, dtype=int
        )
    else:
        # Apply Label Encoding for >= 5 unique categories[cite: 10]
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

print("\n--- Encoded DataFrame Sample ---")
print(df_encoded.head())


--- Encoded DataFrame Sample ---
   age  workclass  fnlwgt  education  education_num  marital_status  \
0   39          6   77516          9             13               4   
1   50          5   83311          9             13               2   
2   38          3  215646         11              9               0   
3   53          3  234721          1              7               2   
4   28          3  338409          9             13               2   

   occupation  relationship  race  capital_gain  capital_loss  hours_per_week  \
0           0             1     4          2174             0              40   
1           3             0     4             0             0              13   
2           5             1     4             0             0              40   
3           5             0     2             0             0              40   
4           9             5     2             0             0              40   

   native_country  sex_ Male  income_ >50K  
0      

# **4. Feature Engineering**

In [14]:
df_fe = df.copy()
df_fe["net_capital"] = df_fe["capital_gain"] - df_fe["capital_loss"]
df_fe["work_hours_age_ratio"] = df_fe["hours_per_week"] / df_fe["age"]

print("\n--- Feature Engineering Sample ---")
print(df_fe[["net_capital", "work_hours_age_ratio"]].head())
df_fe["capital_gain_log"] = np.log1p(df_fe["capital_gain"])

print(
    "\nOriginal Capital Gain Skewness:",
    df_fe["capital_gain"].skew(),
)
print(
    "Log Transformed Capital Gain Skewness:",
    df_fe["capital_gain_log"].skew(),
)


--- Feature Engineering Sample ---
   net_capital  work_hours_age_ratio
0         2174              1.025641
1            0              0.260000
2            0              1.052632
3            0              0.754717
4            0              1.428571

Original Capital Gain Skewness: 11.953847687699799
Log Transformed Capital Gain Skewness: 3.096143524467517
